# CT-RATE Lumora Training (2D Version)

This notebook contains the CT-RATE training flow end to end on pre-converted 2D images: configuration, smoke tests, model definition, two-phase training, and checkpoint saving. All data pipeline helpers have been merged directly into this notebook, removing any external dependency on `ct_rate_utils.py`.

In [1]:
from pathlib import Path

DATA_DIR = Path("data/ct_rate")
CHECKPOINT_DIR = Path("checkpoints/ct_rate")
FINAL_MODEL_NAME = "ct_rate_vlm_phase2_fully_trained.pt"

BATCH_SIZE = 4
MAX_TEXT_LENGTH = 128
NUM_WORKERS = 0
PHASE1_EPOCHS = 3
PHASE2_EPOCHS = 3
PHASE1_LR = 1e-4
PHASE2_LR = 2e-5
LIMIT_TRAIN = None
LIMIT_VAL = None
SAVE_EPOCH_CHECKPOINTS = False
SAVE_PHASE1_FINAL = False
SAVE_OPTIMIZER = False
RESUME_CHECKPOINT = None

print("Configuration loaded")
print(f"DATA_DIR: {DATA_DIR.resolve()}")
print(f"CHECKPOINT_DIR: {CHECKPOINT_DIR.resolve()}")

Configuration loaded
DATA_DIR: /Users/md.nurealamsiddiquee/Projects/lumora/data/ct_rate
CHECKPOINT_DIR: /Users/md.nurealamsiddiquee/Projects/lumora/checkpoints/ct_rate


## Download Check

The dataset download and conversion to 2D images are assumed to be completed already. Below is a check cell for the dataset folder structure.

In [2]:
# Verify that the 2D images directory exists
images_dir = DATA_DIR / "images"
if images_dir.exists():
    print(f"Found 2D images directory under {images_dir.resolve()}")
else:
    print(f"WARNING: 2D images directory not found at {images_dir.resolve()}. Please ensure images are converted and stored in the images/ split folder.")

Found 2D images directory under /Users/md.nurealamsiddiquee/Projects/lumora/data/ct_rate/images


## Data Pipeline Smoke Test

This cell checks that the pre-converted 2D PNG images match the report CSV rows and load correctly into the PyTorch dataset.

In [3]:
import os
from pathlib import Path
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from transformers import AutoTokenizer

REPORT_CSV_CANDIDATES = {
    "train": [
        "dataset/radiology_text_reports/train_reports.csv",
        "radiology_text_reports/train_reports.csv",
        "train_reports.csv",
    ],
    "valid": [
        "dataset/radiology_text_reports/validation_reports.csv",
        "dataset/radiology_text_reports/valid_reports.csv",
        "radiology_text_reports/validation_reports.csv",
        "radiology_text_reports/valid_reports.csv",
        "validation_reports.csv",
        "valid_reports.csv",
    ],
}

def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

def first_existing_path(data_dir, candidates):
    data_dir = Path(data_dir)
    for candidate in candidates:
        path = data_dir / candidate
        if path.exists():
            return path
    return None

def read_report_csv(data_dir, split):
    path = first_existing_path(data_dir, REPORT_CSV_CANDIDATES[split])
    if path is None:
        raise FileNotFoundError(
            f"Could not find the CT-RATE {split} report CSV."
        )
    df = pd.read_csv(path)
    df["__split"] = split
    return df, path

def find_volume_column(df):
    candidates = [
        "VolumeName",
        "volume_name",
        "volume",
        "Volume",
        "FileName",
        "filename",
        "Image",
        "image",
    ]
    for column in candidates:
        if column in df.columns:
            return column
    nii_columns = [
        column
        for column in df.columns
        if df[column].astype(str).str.contains(r"\.(nii(\.gz)?|png)$", regex=True, na=False).any()
    ]
    if nii_columns:
        return nii_columns[0]
    raise ValueError(f"Could not infer a CT-RATE volume filename column from: {list(df.columns)}")

def report_text(row):
    preferred = [
        "Findings_EN",
        "Impressions_EN",
        "Findings",
        "Impressions",
        "Report",
        "report",
        "Text",
        "text",
    ]
    parts = []
    for column in preferred:
        if column in row.index and pd.notna(row[column]) and str(row[column]).strip():
            parts.append(str(row[column]).strip())
    if not parts:
        ignored = {"__split"}
        for column in row.index:
            if column in ignored:
                continue
            value = row[column]
            if pd.notna(value) and isinstance(value, str) and len(value.strip()) > 20:
                parts.append(value.strip())
    if not parts:
        return "Findings: No radiology report text is available."
    return " ".join(parts)

def resolve_image_path(path_str, data_dir):
    if not path_str:
        return None
    p = Path(path_str)
    if p.exists():
        return p
    parts = p.parts
    for i in range(len(parts)):
        if parts[i] in ("images", "dataset", "data"):
            sub_path = Path(*parts[i:])
            if parts[i] == "data" and i + 1 < len(parts) and parts[i+1] == "ct_rate":
                sub_path = Path(*parts[i+2:])
            trial = data_dir / sub_path
            if trial.exists():
                return trial
    return p

def build_entries(data_dir, split, csv_path=None):
    if csv_path is None:
        df, csv_path = read_report_csv(data_dir, split)
    else:
        csv_path = Path(csv_path)
        df = pd.read_csv(csv_path)
        df["__split"] = split
    
    volume_column = find_volume_column(df)
    has_image_path = "image_path" in df.columns
    
    entries = []
    for _, row in df.iterrows():
        volume_name = str(row[volume_column]).strip()
        if not volume_name or volume_name.lower() == "nan":
            continue
        
        image_path = ""
        if has_image_path:
            raw = str(row.get("image_path", "")).strip()
            if raw and raw.lower() != "nan":
                image_path = raw
        
        if not image_path:
            png_name = volume_name
            if png_name.endswith(".nii.gz"):
                png_name = png_name.removesuffix(".nii.gz") + ".png"
            elif not png_name.endswith(".png"):
                png_name = png_name + ".png"
            image_path = str(data_dir / "images" / split / png_name)
            
        resolved_p = resolve_image_path(image_path, data_dir)
        if resolved_p and resolved_p.exists():
            entries.append(
                {
                    "split": split,
                    "volume_name": volume_name,
                    "image_path": str(resolved_p),
                    "text": report_text(row),
                }
            )
    if not entries:
        raise ValueError(f"No usable CT-RATE report rows found in {csv_path} with existing local images")
    return entries, csv_path

class CTRateReportDataset(Dataset):
    def __init__(self, entries, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.entries = entries
        if not self.entries:
            raise ValueError("No local CT-RATE images matched the report CSV.")
            
        self.transform = transforms.Compose(
            [
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225],
                ),
            ]
        )

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]
        try:
            image = Image.open(entry["image_path"]).convert("RGB")
            image = self.transform(image)
        except Exception:
            image = torch.zeros(3, 224, 224)

        tokens = self.tokenizer(
            entry["text"],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        input_ids = tokens["input_ids"].squeeze(0)
        attention_mask = tokens["attention_mask"].squeeze(0)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "image": image,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

device = get_device()
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

train_entries, train_csv = build_entries(DATA_DIR, "train")
train_dataset = CTRateReportDataset(train_entries[:8], tokenizer, MAX_TEXT_LENGTH)
sample = train_dataset[0]

print(f"Device: {device}")
print(f"Train CSV: {train_csv}")
print(f"Train samples with local files: {len(train_dataset):,}")
print(f"Image tensor shape: {tuple(sample['image'].shape)}")
print(f"Input IDs shape: {tuple(sample['input_ids'].shape)}")

Device: mps
Train CSV: data/ct_rate/dataset/radiology_text_reports/train_reports.csv
Train samples with local files: 8
Image tensor shape: (3, 224, 224)
Input IDs shape: (128,)


## Model And Training Helpers

In [4]:
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
from tqdm.auto import tqdm
from transformers import GPT2LMHeadModel

class MedicalReportGenerator(nn.Module):
    def __init__(self, vocab_size=50257, embed_dim=768):
        super().__init__()
        self.encoder = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        num_ftrs = self.encoder.classifier.in_features
        self.encoder.classifier = nn.Identity()
        self.projector = nn.Linear(num_ftrs, embed_dim)
        self.decoder = GPT2LMHeadModel.from_pretrained("gpt2")
        self.decoder.resize_token_embeddings(vocab_size)

    def forward(self, images, input_ids, attention_mask):
        visual_features = self.encoder(images)
        visual_embeddings = self.projector(visual_features).unsqueeze(1)
        text_embeddings = self.decoder.transformer.wte(input_ids)
        inputs_embeds = torch.cat((visual_embeddings, text_embeddings), dim=1)

        visual_mask = torch.ones((images.size(0), 1), device=images.device)
        extended_mask = torch.cat((visual_mask, attention_mask), dim=1)
        return self.decoder(inputs_embeds=inputs_embeds, attention_mask=extended_mask).logits

def set_phase(model, phase):
    if phase == "warmup":
        print("Phase 1: encoder frozen, training projector and GPT-2.")
        for p in model.encoder.parameters():
            p.requires_grad = False
        for p in model.projector.parameters():
            p.requires_grad = True
        for p in model.decoder.parameters():
            p.requires_grad = True
    elif phase == "joint":
        print("Phase 2: all layers unfrozen.")
        for p in model.parameters():
            p.requires_grad = True
    else:
        raise ValueError(f"Unknown phase: {phase}")
    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    return filter(lambda p: p.requires_grad, model.parameters())

def run_epoch(model, loader, optimizer, criterion, device, phase_label, train=True):
    model.train(train)
    total_loss = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()

    with ctx:
        bar = tqdm(loader, desc=phase_label, leave=True)
        for batch in bar:
            images = batch["image"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            if train:
                optimizer.zero_grad()

            logits = model(images, input_ids, attention_mask)
            shift_logits = logits[:, 1:-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            bar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / max(len(loader), 1)

def save_checkpoint(model, optimizer, epoch, train_loss, val_loss, phase1_done, path):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "phase1_done": phase1_done,
        },
        path,
    )
    print(f"Checkpoint saved: {path}")

def require_training_files(data_dir, train_csv=None, valid_csv=None):
    train_path = train_csv if train_csv else first_existing_path(data_dir, REPORT_CSV_CANDIDATES["train"])
    valid_path = valid_csv if valid_csv else first_existing_path(data_dir, REPORT_CSV_CANDIDATES["valid"])
    missing = []
    if train_path is None or not Path(train_path).exists():
        missing.append("train_reports.csv")
    if valid_path is None or not Path(valid_path).exists():
        missing.append("validation_reports.csv")
    if missing:
        lines = "\n  - ".join(missing)
        raise SystemExit(
            "CT-RATE is not ready for training because required report CSVs are missing:\n"
            f"  - {lines}"
        )

def build_loaders(data_dir, tokenizer, device, train_csv=None, valid_csv=None, limit_train=None, limit_val=None):
    from pathlib import Path as _Path
    train_entries, train_csv_path = build_entries(data_dir, "train", train_csv)
    val_entries, val_csv_path = build_entries(data_dir, "valid", valid_csv)

    print(f"Train entries with local files: {len(train_entries):,}")
    print(f"Valid entries with local files: {len(val_entries):,}")

    if not train_entries:
        raise RuntimeError("No local train images found.")
    if not val_entries:
        raise RuntimeError("No local validation images found.")

    if limit_train:
        train_entries = train_entries[:limit_train]
    if limit_val:
        val_entries = val_entries[:limit_val]

    train_dataset = CTRateReportDataset(train_entries, tokenizer, MAX_TEXT_LENGTH)
    val_dataset = CTRateReportDataset(val_entries, tokenizer, MAX_TEXT_LENGTH)

    print(f"Train CSV: {_Path(train_csv_path).resolve()}")
    print(f"Valid CSV: {_Path(val_csv_path).resolve()}")

    pin = device.type == "cuda"
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=pin,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin,
    )
    return train_dataset, val_dataset, train_loader, val_loader

## Train

Set `LIMIT_TRAIN` and `LIMIT_VAL` to `None` in the configuration cell when you are ready to train on all downloaded files.

In [ ]:
require_training_files(DATA_DIR)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

device = get_device()
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

train_dataset, val_dataset, train_loader, val_loader = build_loaders(
    DATA_DIR,
    tokenizer,
    device,
    limit_train=LIMIT_TRAIN,
    limit_val=LIMIT_VAL,
)
print(f"Train samples: {len(train_dataset):,} ({len(train_loader):,} batches)")
print(f"Val samples: {len(val_dataset):,} ({len(val_loader):,} batches)")

sample = next(iter(train_loader))
print(f"Sample image tensor: {tuple(sample['image'].shape)}")
print(f"Sample input ids: {tuple(sample['input_ids'].shape)}")

model = MedicalReportGenerator(vocab_size=len(tokenizer)).to(device)
resume_epoch = 0
phase1_done = False
saved_optimizer = None

if RESUME_CHECKPOINT and Path(RESUME_CHECKPOINT).exists():
    checkpoint = torch.load(RESUME_CHECKPOINT, map_location=device)
    state = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
    model.load_state_dict(state)
    if isinstance(checkpoint, dict):
        resume_epoch = checkpoint.get("epoch", 0)
        phase1_done = checkpoint.get("phase1_done", False)
        saved_optimizer = checkpoint.get("optimizer_state_dict")
    print(f"Resumed checkpoint: {RESUME_CHECKPOINT}")

criterion = nn.CrossEntropyLoss(ignore_index=-100)

if phase1_done:
    print("Phase 1 already complete; skipping warmup.")
else:
    trainable = set_phase(model, "warmup")
    optimizer = torch.optim.AdamW(trainable, lr=PHASE1_LR)
    if saved_optimizer is not None and resume_epoch < PHASE1_EPOCHS:
        optimizer.load_state_dict(saved_optimizer)

    train_loss = math.nan
    val_loss = math.nan
    for epoch in range(resume_epoch + 1, PHASE1_EPOCHS + 1):
        train_loss = run_epoch(model, train_loader, optimizer, criterion, device, f"Train P1 {epoch}")
        val_loss = run_epoch(model, val_loader, optimizer, criterion, device, f"Val P1 {epoch}", train=False)
        if SAVE_EPOCH_CHECKPOINTS:
            save_checkpoint(
                model,
                optimizer,
                epoch,
                train_loss,
                val_loss,
                phase1_done=False,
                path=CHECKPOINT_DIR / f"ct_rate_vlm_phase1_epoch{epoch}_checkpoint.pt",
            )

    if SAVE_PHASE1_FINAL:
        save_checkpoint(
            model,
            optimizer,
            PHASE1_EPOCHS,
            train_loss,
            val_loss,
            phase1_done=True,
            path=CHECKPOINT_DIR / "ct_rate_vlm_phase1_FINAL.pt",
        )

trainable = set_phase(model, "joint")
optimizer = torch.optim.AdamW(trainable, lr=PHASE2_LR)
avg_train_loss = math.nan
avg_val_loss = math.nan

for epoch in range(1, PHASE2_EPOCHS + 1):
    avg_train_loss = run_epoch(model, train_loader, optimizer, criterion, device, f"Train P2 {epoch}")
    avg_val_loss = run_epoch(model, val_loader, optimizer, criterion, device, f"Val P2 {epoch}", train=False)
    if SAVE_EPOCH_CHECKPOINTS:
        save_checkpoint(
            model,
            optimizer,
            epoch,
            avg_train_loss,
            avg_val_loss,
            phase1_done=True,
            path=CHECKPOINT_DIR / f"ct_rate_vlm_phase2_epoch{epoch}_checkpoint.pt",
        )

final_path = CHECKPOINT_DIR / FINAL_MODEL_NAME
checkpoint = {
    "model_state_dict": model.state_dict(),
    "final_train_loss": avg_train_loss,
    "final_val_loss": avg_val_loss,
    "phase1_done": True,
    "config": {
        "data_dir": str(DATA_DIR),
        "checkpoint_dir": str(CHECKPOINT_DIR),
        "batch_size": BATCH_SIZE,
        "max_text_length": MAX_TEXT_LENGTH,
        "num_workers": NUM_WORKERS,
        "phase1_epochs": PHASE1_EPOCHS,
        "phase1_lr": PHASE1_LR,
        "phase2_epochs": PHASE2_EPOCHS,
        "phase2_lr": PHASE2_LR,
        "limit_train": LIMIT_TRAIN,
        "limit_val": LIMIT_VAL,
    },
}
if SAVE_OPTIMIZER:
    checkpoint["optimizer_state_dict"] = optimizer.state_dict()
torch.save(checkpoint, final_path)
print(f"Final model saved: {final_path.resolve()}")